In [1]:
from ase.io import read, write
from mattersim.forcefield import Potential
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

potential = Potential.from_checkpoint(load_path='./hard_const/results/best_model.pth', device='cuda', long_range = True)


/global/cfs/cdirs/m4555/Jun/mattersim-LR/src/mattersim/__version__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/global/cfs/projectdirs/m4555/Jun/conda_envs/mattersim-LR/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2025-11-15 17:16:19.299 | INFO     | mattersim.forcefield.potential:from_checkpoint:908 - Loading the model from ./hard_const/results/best_model.pth


In [22]:
from mattersim.utils.atoms_utils import AtomsAdaptor
from mattersim.datasets.utils.build import build_dataloader

atoms_train = AtomsAdaptor.from_file("./solvation_zn/zn_300_100.000ps.xyz")
energies, forces = [], []

for atoms in atoms_train:
    energies.append(atoms.get_potential_energy())
    forces.append(atoms.get_forces())

dataloader = build_dataloader(
    atoms_train,
    energies,
    forces,
    shuffle=False,
    pin_memory=False,
    batch_size=8
)

In [23]:
predicted_energies, predicted_forces, predicted_stress, predicted_charges = potential.predict_properties(
    dataloader,
    include_forces=False,
    include_stresses=False,
)

In [24]:
import numpy as np
from ase.io import write

t = 0
for at in atoms_train:
    n = len(at)
    chunk = predicted_charges[t:t+n]        # this is a list
    chunk = np.asarray(chunk, dtype=float)  # make it a NumPy array

    at.new_array('charge', chunk)           # now works
    t += n

write("zn_charges.extxyz", atoms_train, format='extxyz')
